<a href="https://colab.research.google.com/github/rudalshan0412-code/attention-is-all-you-need-pytorch/blob/main/03)_Positional_Encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive 연결

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 프로젝트 경로 설정

from pathlib import Path
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/attention_is_all_you_need")
SRC_DIR = PROJECT_ROOT / "src"

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: /content/drive/MyDrive/attention_is_all_you_need
SRC_DIR: /content/drive/MyDrive/attention_is_all_you_need/src


In [ ]:
# 기존 파일 구조 확인

for path in sorted(SRC_DIR.iterdir()):
    print(path.name)

__pycache__
attention.py
multi_head_attention.py


In [ ]:
'''
Positional Encoding의 역할

Attention은 token의 순서 자체를 알지 못한다. 즉, 순서정보를 더해주지 않으면 "I love you"와 "you love I"의 attention 결과가 동일하게 된다.
따라서, 데이터에 순서 정보를 더해줄 필요가 있다.

이때, sin과 cos을 이용해 표현하는데, PE(pos, 2i) = sin(pos/10000^2i/dmodel)와 PE(pos, 2i+1) = cos(pos/10000^2i/dmodel)이다.
(d model은 무조건 짝수이다.)

sin과 cos을 사용하는 이유는 삼각함수의 가법정리(sin(a+b) = sin a * cos b + cos a* sin b)로 인해 PE(pos)·PE(pos + k)(내적) = cos wk 로 나오기 때문이다. 이를 통해 상대적인 거리를 알 수 있다.(코사인의 비선형성으로 인해 가까울때는 가파르게, 멀수록 완만하게 작동하게 된다).

이때 PE(pos)의 벡터 수는 token의 개수와 일치하며, 차원 수는 임베딩된 토큰의 차원 수를 따라간다.

예를 들어, d model = 4, 임베딩된 차원수가 4개인 "I love food"의 경우, pos는 인덱스를 의미함으로 0, 1, 2가 있으며, 차원수가 4개임으로 i는 2개이다. pos = 0 인 경우[sin0/10000^((2*0)/4)), cos0/10000^((2*0)/4), sin0/10000((2*1)/4), cos0/10000((2*1)/4)]가 된다.

+ positional encoding은 위치정보를 concatenation하지 않고 기존 임베딩된 데이터에 더하게 되는데, 이는 기존 d_model의 shape를 유지할 수 있기 때문이다.
'''

'\nPositional Encoding의 역할\n\nAttention은 token의 순서 자체를 알지 못한다. 즉, 순서정보를 더해주지 않으면 "I love you"와 "you love I"의 attention 결과가 동일하게 된다.\n따라서, 데이터에 순서 정보를 더해줄 필요가 있다.\n\n이때, sin과 cos을 이용해 표현하는데, PE(pos, 2i) = sin(pos/10000^2i/dmodel)와 PE(pos, 2i+1) = cos(pos/10000^2i/dmodel)이다.\n(d model은 무조건 짝수이다.)\n\nsin과 cos을 사용하는 이유는 삼각함수의 가법정리(sin(a+b) = sin a * cos b + cos a* sin b)로 인해 PE(pos)·PE(pos + k)(내적) = cos wk 로 나오기 때문이다. 이를 통해 상대적인 거리를 알 수 있다.(코사인의 비선형성으로 인해 가까울때는 가파르게, 멀수록 완만하게 작동하게 된다).\n\n이때 PE(pos)의 벡터 수는 token의 개수와 일치하며, 차원 수는 임베딩된 토큰의 차원 수를 따라간다.\n\n예를 들어, d model = 4, 임베딩된 차원수가 4개인 "I love food"의 경우, pos는 인덱스를 의미함으로 0, 1, 2가 있으며, 차원수가 4개임으로 i는 2개이다. pos = 0 인 경우[sin0/10000^((2*0)/4)), cos0/10000^((2*0)/4), sin0/10000((2*1)/4), cos0/10000((2*1)/4)]가 된다.\n\n'

In [ ]:
'''
pos: 인덱스
i: 차원수의 1/2
'''

In [ ]:
# position과 div_term 작은 예제로 만들어보기
# 10000^2i/dmodel이 div_term에 해당함
import math
import torch

d_model = 4
max_len = 5

position = torch.arange(
    max_len,
    dtype=torch.float32, # 0부터 max_len-1 까지 1씩 커지는 tensor 생성
).unsqueeze(1) # 1번 위치에 차원 추가
# position은 인덱스(pos)가 세로로 늘어진 열벡터
print(position)
print("position shape:", position.shape)

tensor([[0.],
        [1.],
        [2.],
        [3.],
        [4.]])
position shape: torch.Size([5, 1])


In [ ]:
#div_term 만들기
# div_term은 주기 조절용 스케일링 값
div_term = torch.exp( # 모든 값들을 e^x의 x에 대입함
    torch.arange(
        0,
        d_model, # 위에서의 예제에서 d_model = 4로 선언되어 있다.
        2,
        dtype=torch.float32,
    ) # 0부터 d_model까지 2씩 증가
    * (-math.log(10000.0) / d_model)
)
# torch.arrange(0, d_model, 2, dtype = torch.float32)가 2i의 역할을 한다
# a^x = exp(xlog(a))라고 놓을 수 있는데, 따라서 pos * 10000^(-2i / d_model)는 -math.log(10000) / d_model 이라고 놓을 수 있다.

print(div_term)
print("div_term shape:", div_term.shape)

tensor([1.0000, 0.0100])
div_term shape: torch.Size([2])


In [ ]:
# Broadcasting 직접 확인
# Broadcasting은 위치 * 주기의 모든 값에 대한 matrix를 한번에 생성
# 둘이 곱해질 수 있는지 확인

angles = position * div_term

print(angles)
print("angles shape:", angles.shape)

tensor([[0.0000, 0.0000],
        [1.0000, 0.0100],
        [2.0000, 0.0200],
        [3.0000, 0.0300],
        [4.0000, 0.0400]])
angles shape: torch.Size([5, 2])


In [ ]:
# 작은 Positional Encoding Matrix 생성

pe = torch.zeros(max_len, d_model) # 모든 요소가 0으로 채워진 tensor 생

pe[:, 0::2] = torch.sin(position * div_term) # pe[:, 0::2] :(모든 행)에 대해 0::2(0부터 시작해서 2씩 건너뛰면서) 선택하겠다
pe[:, 1::2] = torch.cos(position * div_term)

print(pe)
print("PE shape:", pe.shape)


tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  0.9999],
        [ 0.9093, -0.4161,  0.0200,  0.9998],
        [ 0.1411, -0.9900,  0.0300,  0.9996],
        [-0.7568, -0.6536,  0.0400,  0.9992]])
PE shape: torch.Size([5, 4])


In [ ]:
# Positional Encoding 전체 코드

import math

import torch
import torch.nn as nn


class PositionalEncoding(nn.Module):
    """
    Sinusoidal Positional Encoding

    Input:
        (batch_size, seq_len, d_model)

    Output:
        (batch_size, seq_len, d_model)
    """

    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()

        if d_model % 2 != 0:
            raise ValueError("현재 구현에서는 d_model이 짝수여야 합니다.")

        self.d_model = d_model
        self.max_len = max_len

        self.dropout = nn.Dropout(dropout) # token 정보와 position 정보가 결합된 transformer 입력 전체에 dropout 적용

        # position:
        # (max_len, 1)
        position = torch.arange(
            max_len,
            dtype=torch.float32,
        ).unsqueeze(1)

        # div_term:
        # (d_model / 2,)
        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32,
            )
            * (-math.log(10000.0) / d_model)
        )

        # pe:
        # (max_len, d_model)
        pe = torch.zeros(
            max_len,
            d_model,
        )

        # 짝수 dimension → sin
        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        # 홀수 dimension → cos
        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        # batch dimension 추가
        #
        # (max_len, d_model)
        # →
        # (1, max_len, d_model)
        pe = pe.unsqueeze(0) # 0번 위치에 크기가 1인 벡터 추가
        # (1, max_len, d_model) 형태로 바꿔주는 이유: 어차피 모든 batch 마다 위치정보가 추가되어야하는데,
        #이때 추가되는 위치정보는 모두 동일함(어떤 batch가 들어오든 간에).
        # 따라서, 하나만 만들어 놓은 다음, broadcasting을 통해 재사용함

        self.register_buffer( # tensor 등록
            "pe",
            pe,
        )
        # Positional Encoding은 학습되는 weight가 아니기 때문에 nn.Parameter(모델이 학습해야할 가중치나 편향이라고 지정)를 만들 필요 없다.

    def forward(self, x):
        """
        Args:
            x:
                (batch_size, seq_len, d_model)

        Returns:
            output:
                (batch_size, seq_len, d_model)
        """

        seq_len = x.size(1)

        if x.size(-1) != self.d_model:
            raise ValueError(
                f"x의 마지막 차원은 d_model={self.d_model}이어야 합니다. "
                f"현재 값: {x.size(-1)}"
            )

        if seq_len > self.max_len:
            raise ValueError(
                f"seq_len={seq_len}이 max_len={self.max_len}보다 큽니다."
            )

        x = x + self.pe[:, :seq_len, :]

        output = self.dropout(x)

        return output

In [ ]:
# 코드 src/positional_encodin.py로 저장

%%writefile /content/drive/MyDrive/attention_is_all_you_need/src/positional_encoding.py

import math

import torch
import torch.nn as nn


class PositionalEncoding(nn.Module):
    """
    Sinusoidal Positional Encoding

    Input:
        (batch_size, seq_len, d_model)

    Output:
        (batch_size, seq_len, d_model)
    """

    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()

        if d_model % 2 != 0:
            raise ValueError("현재 구현에서는 d_model이 짝수여야 합니다.")

        self.d_model = d_model
        self.max_len = max_len

        self.dropout = nn.Dropout(dropout)

        position = torch.arange(
            max_len,
            dtype=torch.float32,
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
                dtype=torch.float32,
            )
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            max_len,
            d_model,
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe,
        )

    def forward(self, x):
        """
        Args:
            x:
                (batch_size, seq_len, d_model)

        Returns:
            output:
                (batch_size, seq_len, d_model)
        """

        seq_len = x.size(1)

        if x.size(-1) != self.d_model:
            raise ValueError(
                f"x의 마지막 차원은 d_model={self.d_model}이어야 합니다. "
                f"현재 값: {x.size(-1)}"
            )

        if seq_len > self.max_len:
            raise ValueError(
                f"seq_len={seq_len}이 max_len={self.max_len}보다 큽니다."
            )

        x = x + self.pe[:, :seq_len, :]

        output = self.dropout(x)

        return output

Writing /content/drive/MyDrive/attention_is_all_you_need/src/positional_encoding.py


In [ ]:
# 파일 구조 다시 확인
for path in sorted(SRC_DIR.iterdir()):
    print(path.name)

__pycache__
attention.py
multi_head_attention.py
positional_encoding.py


In [ ]:
# 저장한 class import

from src.positional_encoding import PositionalEncoding

print("PositionalEncoding import 성공")

PositionalEncoding import 성공


In [ ]:
# shape 확인용 테스트
# 정확한 PE 값을 확인하기 위해 dropout = 0.0으로 설정

import torch

batch_size = 2
seq_len = 4
d_model = 8
max_len = 10

positional_encoding = PositionalEncoding(
    d_model=d_model,
    max_len=max_len,
    dropout=0.0,
)

x = torch.zeros(
    batch_size,
    seq_len,
    d_model,
)

output = positional_encoding(x)

print("입력 x shape:")
print(x.shape)

print("\n전체 PE buffer shape:")
print(positional_encoding.pe.shape)

current_pe = positional_encoding.pe[:, :seq_len, :]

print("\n현재 sequence 길이만큼 자른 PE shape:")
print(current_pe.shape)

added = x + current_pe

print("\nx + PE shape:")
print(added.shape)

print("\n최종 output shape:")
print(output.shape)


입력 x shape:
torch.Size([2, 4, 8])

전체 PE buffer shape:
torch.Size([1, 10, 8])

현재 sequence 길이만큼 자른 PE shape:
torch.Size([1, 4, 8])

x + PE shape:
torch.Size([2, 4, 8])

최종 output shape:
torch.Size([2, 4, 8])


In [ ]:
# output shape 자동 검증

assert output.shape == x.shape

print("출력 shape 테스트 통과")

출력 shape 테스트 통과


In [ ]:
# position = 0 테스트

print(output[0, 0])

tensor([0., 1., 0., 1., 0., 1., 0., 1.])


In [ ]:
# sequence 위치가 다르면 값도 달라지는지 확인

print("position 0:")
print(output[0, 0])

print("\nposition 1:")
print(output[0, 1])

print("\nposition 2:")
print(output[0, 2])

position 0:
tensor([0., 1., 0., 1., 0., 1., 0., 1.])

position 1:
tensor([0.8415, 0.5403, 0.0998, 0.9950, 0.0100, 0.9999, 0.0010, 1.0000])

position 2:
tensor([ 0.9093, -0.4161,  0.1987,  0.9801,  0.0200,  0.9998,  0.0020,  1.0000])


In [ ]:
# batch broadcasting 데스트

# 위치는 batch와 무관하기에

print("batch 0, position 2:")
print(output[0, 2])

print("\nbatch 1, position 2:")
print(output[1, 2])

# 두 값이 동일해야한다

batch 0, position 2:
tensor([ 0.9093, -0.4161,  0.1987,  0.9801,  0.0200,  0.9998,  0.0020,  1.0000])

batch 1, position 2:
tensor([ 0.9093, -0.4161,  0.1987,  0.9801,  0.0200,  0.9998,  0.0020,  1.0000])


In [ ]:
# 짝수 dimension = sin, 홀수는 cos 인지 확인

position = torch.arange(
    max_len,
    dtype=torch.float32,
).unsqueeze(1)

div_term = torch.exp(
    torch.arange(
        0,
        d_model,
        2,
        dtype=torch.float32,
    )
    * (-torch.log(torch.tensor(10000.0)) / d_model)
)

expected_sin = torch.sin(
    position[:seq_len] * div_term
)

expected_cos = torch.cos(
    position[:seq_len] * div_term
)

actual_pe = positional_encoding.pe[
    0,
    :seq_len,
    :
]

print("실제 PE의 짝수 dimensions:")
print(actual_pe[:, 0::2])

print("\n직접 계산한 sin:")
print(expected_sin)

print("\n실제 PE의 홀수 dimensions:")
print(actual_pe[:, 1::2])

print("\n직접 계산한 cos:")
print(expected_cos)

실제 PE의 짝수 dimensions:
tensor([[0.0000, 0.0000, 0.0000, 0.0000],
        [0.8415, 0.0998, 0.0100, 0.0010],
        [0.9093, 0.1987, 0.0200, 0.0020],
        [0.1411, 0.2955, 0.0300, 0.0030]])

직접 계산한 sin:
tensor([[0.0000, 0.0000, 0.0000, 0.0000],
        [0.8415, 0.0998, 0.0100, 0.0010],
        [0.9093, 0.1987, 0.0200, 0.0020],
        [0.1411, 0.2955, 0.0300, 0.0030]])

실제 PE의 홀수 dimensions:
tensor([[ 1.0000,  1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.9950,  0.9999,  1.0000],
        [-0.4161,  0.9801,  0.9998,  1.0000],
        [-0.9900,  0.9553,  0.9996,  1.0000]])

직접 계산한 cos:
tensor([[ 1.0000,  1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.9950,  0.9999,  1.0000],
        [-0.4161,  0.9801,  0.9998,  1.0000],
        [-0.9900,  0.9553,  0.9996,  1.0000]])


In [ ]:
# d_model = 4 PE matrix 재확인

small_pe = PositionalEncoding(
    d_model=4,
    max_len=5,
    dropout=0.0,
)

for pos in range(5):
    print(
        f"position {pos}:",
        small_pe.pe[0, pos]
    )

position 0: tensor([0., 1., 0., 1.])
position 1: tensor([0.8415, 0.5403, 0.0100, 0.9999])
position 2: tensor([ 0.9093, -0.4161,  0.0200,  0.9998])
position 3: tensor([ 0.1411, -0.9900,  0.0300,  0.9996])
position 4: tensor([-0.7568, -0.6536,  0.0400,  0.9992])


In [ ]:
# register_buffer() 가 Parameter가 아닌지 확인

model = PositionalEncoding(
    d_model=8,
    max_len=10,
    dropout=0.1,
)

print("Parameters:")
for name, parameter in model.named_parameters():
    print(name)

print("\nBuffers:")
for name, buffer in model.named_buffers():
    print(name, buffer.shape)

Parameters:

Buffers:
pe torch.Size([1, 10, 8])


In [ ]:
# 최종 통합 테스트

import torch

from src.positional_encoding import PositionalEncoding


batch_size = 2
seq_len = 4
d_model = 8
max_len = 10


model = PositionalEncoding(
    d_model=d_model,
    max_len=max_len,
    dropout=0.0,
)

x = torch.zeros(
    batch_size,
    seq_len,
    d_model,
)

output = model(x)


# 1. 클래스 실행 및 output shape
assert output.shape == (
    batch_size,
    seq_len,
    d_model,
)


# 2. 전체 PE buffer shape
assert model.pe.shape == (
    1,
    max_len,
    d_model,
)


# 3. 같은 position은 batch가 달라도 같은 PE
assert torch.allclose(
    output[0, 2],
    output[1, 2],
)


# 4. 다른 position은 다른 PE
assert not torch.allclose(
    output[0, 0],
    output[0, 1],
)


# 5. position=0
expected_position_0 = torch.tensor(
    [0., 1., 0., 1., 0., 1., 0., 1.]
)

assert torch.allclose(
    output[0, 0],
    expected_position_0,
)


# 6. sin / cos 직접 검증
position = torch.arange(
    max_len,
    dtype=torch.float32,
).unsqueeze(1)

div_term = torch.exp(
    torch.arange(
        0,
        d_model,
        2,
        dtype=torch.float32,
    )
    * (-torch.log(torch.tensor(10000.0)) / d_model)
)

expected_sin = torch.sin(
    position[:seq_len] * div_term
)

expected_cos = torch.cos(
    position[:seq_len] * div_term
)

actual_pe = model.pe[
    0,
    :seq_len,
    :
]

assert torch.allclose(
    actual_pe[:, 0::2],
    expected_sin,
)

assert torch.allclose(
    actual_pe[:, 1::2],
    expected_cos,
)


print("모든 Positional Encoding 테스트 통과")
print()
print("input shape :", x.shape)
print("PE shape    :", model.pe.shape)
print("output shape:", output.shape)

print("\nposition 0:")
print(output[0, 0])

print("\nposition 1:")
print(output[0, 1])

모든 Positional Encoding 테스트 통과

input shape : torch.Size([2, 4, 8])
PE shape    : torch.Size([1, 10, 8])
output shape: torch.Size([2, 4, 8])

position 0:
tensor([0., 1., 0., 1., 0., 1., 0., 1.])

position 1:
tensor([0.8415, 0.5403, 0.0998, 0.9950, 0.0100, 0.9999, 0.0010, 1.0000])
